In [2]:
print('test')

test


In [ ]:
# VIDEO_PATH = r"video_real_data/VID_20260302_132743.mp4" - 1500


In [17]:
import cv2
import os
import numpy as np
from pathlib import Path

# ===================== CONFIG =====================
VIDEO_PATH = r"video_real_data/VID_20260302_132743.mp4"
OUTPUT_DIR = r"real_data"
EXTRACT_EVERY_N = 60   # ดึงทุก N เฟรม (1 = ทุกเฟรม, 3 = ทุก 3 เฟรม)
CLASS_ID = 0           # fish class
ZOOM_STEP = 0.2        # ขนาดการซูมแต่ละครั้ง
ZOOM_MAX  = 5.0
ZOOM_MIN  = 1.0
# ==================================================

# ---- Folder structure (YOLO) ----
img_dir = Path(OUTPUT_DIR) / "images" / "train"
lbl_dir = Path(OUTPUT_DIR) / "labels" / "train"
img_dir.mkdir(parents=True, exist_ok=True)
lbl_dir.mkdir(parents=True, exist_ok=True)

# ---- Global state ----
drawing   = False
start_x = start_y = 0
cur_x   = cur_y   = 0
boxes     = []        # [x1,y1,x2,y2] ใน coordinate ของภาพต้นฉบับ
frame_orig = None

frame_index = 0
frames_seen = {}      # frame_index -> (img_path, lbl_path, boxes)

# ---- Zoom state ----
zoom_scale  = 1.0     # zoom ปัจจุบัน
zoom_cx     = 0.5     # จุดศูนย์กลางซูม (0-1) แกน X
zoom_cy     = 0.5     # จุดศูนย์กลางซูม (0-1) แกน Y

WINDOW = "Koi Labeler"


# ============================================================
# Zoom helpers  -  แปลง coordinate ระหว่าง display <-> original
# ============================================================
def get_zoom_region(img_h, img_w):
    """คืน (x1,y1,x2,y2) ของ region ที่แสดงบนหน้าจอ (ใน pixel ต้นฉบับ)"""
    view_w = img_w / zoom_scale
    view_h = img_h / zoom_scale
    cx_px  = zoom_cx * img_w
    cy_px  = zoom_cy * img_h
    x1 = int(np.clip(cx_px - view_w / 2, 0, img_w - view_w))
    y1 = int(np.clip(cy_px - view_h / 2, 0, img_h - view_h))
    x2 = int(x1 + view_w)
    y2 = int(y1 + view_h)
    return x1, y1, x2, y2


def display_to_orig(dx, dy, img_h, img_w, disp_w, disp_h):
    """แปลง coordinate บนหน้าจอ -> coordinate ภาพต้นฉบับ"""
    rx1, ry1, rx2, ry2 = get_zoom_region(img_h, img_w)
    rw = rx2 - rx1
    rh = ry2 - ry1
    ox = int(rx1 + dx / disp_w * rw)
    oy = int(ry1 + dy / disp_h * rh)
    return np.clip(ox, 0, img_w - 1), np.clip(oy, 0, img_h - 1)


def orig_to_display(ox, oy, img_h, img_w, disp_w, disp_h):
    """แปลง coordinate ภาพต้นฉบับ -> coordinate บนหน้าจอ"""
    rx1, ry1, rx2, ry2 = get_zoom_region(img_h, img_w)
    rw = rx2 - rx1
    rh = ry2 - ry1
    dx = int((ox - rx1) / rw * disp_w)
    dy = int((oy - ry1) / rh * disp_h)
    return dx, dy


def apply_zoom(img, disp_w, disp_h):
    """ตัด region แล้ว resize ให้เต็มหน้าต่าง"""
    h, w = img.shape[:2]
    rx1, ry1, rx2, ry2 = get_zoom_region(h, w)
    cropped = img[ry1:ry2, rx1:rx2]
    return cv2.resize(cropped, (disp_w, disp_h), interpolation=cv2.INTER_LINEAR)


# ============================================================
# Mouse callback  (coordinate ที่รับมาคือ display coordinate)
# ============================================================
def mouse_cb(event, x, y, flags, param):
    global drawing, start_x, start_y, cur_x, cur_y, boxes, frame_orig
    global zoom_scale, zoom_cx, zoom_cy

    if frame_orig is None:
        return

    img_h, img_w = frame_orig.shape[:2]
    disp_w, disp_h = param[0], param[1]

    # ---- Scroll wheel = zoom ----
    if event == cv2.EVENT_MOUSEWHEEL:
        if flags > 0:   # scroll up -> zoom in
            zoom_scale = min(zoom_scale + ZOOM_STEP, ZOOM_MAX)
        else:           # scroll down -> zoom out
            zoom_scale = max(zoom_scale - ZOOM_STEP, ZOOM_MIN)
        # เลื่อนจุดศูนย์กลางไปที่ตำแหน่งเมาส์
        ox, oy = display_to_orig(x, y, img_h, img_w, disp_w, disp_h)
        zoom_cx = float(np.clip(ox / img_w, 0.05, 0.95))
        zoom_cy = float(np.clip(oy / img_h, 0.05, 0.95))
        return

    # ---- Middle click drag = pan ----
    if event == cv2.EVENT_MBUTTONDOWN:
        param[2] = (x, y)
        return
    if event == cv2.EVENT_MOUSEMOVE and (flags & cv2.EVENT_FLAG_MBUTTON):
        if param[2] is not None:
            px, py = param[2]
            dx_ratio = (px - x) / disp_w / zoom_scale
            dy_ratio = (py - y) / disp_h / zoom_scale
            zoom_cx = float(np.clip(zoom_cx + dx_ratio, 0.0, 1.0))
            zoom_cy = float(np.clip(zoom_cy + dy_ratio, 0.0, 1.0))
            param[2] = (x, y)
        return

    # ---- LMB = วาด bounding box ----
    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        ox, oy = display_to_orig(x, y, img_h, img_w, disp_w, disp_h)
        start_x, start_y = ox, oy
        cur_x,   cur_y   = ox, oy

    elif event == cv2.EVENT_MOUSEMOVE and drawing:
        ox, oy = display_to_orig(x, y, img_h, img_w, disp_w, disp_h)
        cur_x, cur_y = ox, oy

    elif event == cv2.EVENT_LBUTTONUP and drawing:
        drawing = False
        ox, oy = display_to_orig(x, y, img_h, img_w, disp_w, disp_h)
        x1, y1 = min(start_x, ox), min(start_y, oy)
        x2, y2 = max(start_x, ox), max(start_y, oy)
        if abs(x2 - x1) > 3 and abs(y2 - y1) > 3:
            boxes.append([x1, y1, x2, y2])


# ============================================================
# Draw overlay บน display frame (ซูมแล้ว)
# ============================================================
def draw_overlay(img, frame_idx, total, saved_count, disp_w, disp_h):
    img_h, img_w = img.shape[:2]

    # Apply zoom ก่อน
    vis = apply_zoom(img, disp_w, disp_h)

    # วาดกรอบที่ confirm แล้ว
    for b in boxes:
        dx1, dy1 = orig_to_display(b[0], b[1], img_h, img_w, disp_w, disp_h)
        dx2, dy2 = orig_to_display(b[2], b[3], img_h, img_w, disp_w, disp_h)
        if dx2 > 0 and dy2 > 0 and dx1 < disp_w and dy1 < disp_h:
            cv2.rectangle(vis, (dx1, dy1), (dx2, dy2), (0, 255, 80), 2)
            cv2.putText(vis, "fish", (dx1, max(dy1 - 6, 12)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 80), 1)

    # กรอบที่กำลังวาด
    if drawing:
        dx1, dy1 = orig_to_display(min(start_x, cur_x), min(start_y, cur_y), img_h, img_w, disp_w, disp_h)
        dx2, dy2 = orig_to_display(max(start_x, cur_x), max(start_y, cur_y), img_h, img_w, disp_w, disp_h)
        cv2.rectangle(vis, (dx1, dy1), (dx2, dy2), (0, 200, 255), 2)

    # Zoom indicator (มุมขวาบน)
    if zoom_scale > 1.0:
        zoom_txt = f"ZOOM x{zoom_scale:.1f}"
        (tw, th), _ = cv2.getTextSize(zoom_txt, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(vis, (disp_w - tw - 16, 6), (disp_w - 6, 32), (30, 30, 30), -1)
        cv2.putText(vis, zoom_txt, (disp_w - tw - 10, 26),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 220, 255), 2)

    # Header bar
    cv2.rectangle(vis, (0, 0), (disp_w, 38), (20, 20, 20), -1)
    info = (f"Frame {frame_idx}/{total}  |  Boxes: {len(boxes)}  |  Saved: {saved_count}"
            f"  |  [S/Spc]=Save  [D/->]=Skip  [A/<-]=Back  [Z]=Undo  [Scroll]=Zoom  [Mid+drag]=Pan  [R]=Reset zoom  [Q/ESC]=Quit")
    cv2.putText(vis, info, (8, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (200, 200, 200), 1)

    return vis


# ============================================================
# Save เฟรม + label (บันทึกภาพต้นฉบับ full resolution ทันที)
# ============================================================
def save_frame(img, frame_idx, bxs):
    img_path = img_dir / f"frame_{frame_idx:06d}.jpg"
    lbl_path = lbl_dir / f"frame_{frame_idx:06d}.txt"
    cv2.imwrite(str(img_path), img)   # full res

    h, w = img.shape[:2]
    lines = []
    for b in bxs:
        cx = ((b[0] + b[2]) / 2) / w
        cy = ((b[1] + b[3]) / 2) / h
        bw = (b[2] - b[0]) / w
        bh = (b[3] - b[1]) / h
        lines.append(f"{CLASS_ID} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

    with open(lbl_path, "w") as f:
        f.write("\n".join(lines))

    return str(img_path), str(lbl_path)


# ============================================================
# โหลด label กลับมาเพื่อรองรับ Back
# ============================================================
def load_boxes_from_file(lbl_path, img_w, img_h):
    bxs = []
    if not os.path.exists(lbl_path):
        return bxs
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            _, cx, cy, bw, bh = map(float, parts)
            x1 = int((cx - bw / 2) * img_w)
            y1 = int((cy - bh / 2) * img_h)
            x2 = int((cx + bw / 2) * img_w)
            y2 = int((cy + bh / 2) * img_h)
            bxs.append([x1, y1, x2, y2])
    return bxs


# ============================================================
# สร้าง data.yaml
# ============================================================
def write_yaml():
    yaml_path = Path(OUTPUT_DIR) / "data.yaml"
    content = (
        f"path: {OUTPUT_DIR}\n"
        f"train: images/train\n"
        f"val:   images/train\n\n"
        f"nc: 1\n"
        f"names: ['fish']\n"
    )
    with open(yaml_path, "w") as f:
        f.write(content)
    print(f"[OK] data.yaml -> {yaml_path}")


# ============================================================
# MAIN
# ============================================================
def main():
    global boxes, frame_orig, frame_index
    global zoom_scale, zoom_cx, zoom_cy
    global drawing, start_x, start_y, cur_x, cur_y

    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        print(f"[X] เปิดวิดีโอไม่ได้: {VIDEO_PATH}")
        return

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    saved_count  = 0

    cv2.namedWindow(WINDOW, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(WINDOW, 1280, 720)

    # mouse_param = [disp_w, disp_h, pan_start]
    mouse_param = [1280, 720, None]
    cv2.setMouseCallback(WINDOW, mouse_cb, mouse_param)

    # ---- frame cache ----
    frame_cache = {}

    # ---- ล้าง state ทั้งหมดให้สะอาด (ป้องกัน box เก่าหลุดมา) ----
    global boxes, frames_seen
    boxes       = []
    frames_seen = {}

    def get_frame(idx):
        if idx in frame_cache:
            return frame_cache[idx]
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, f = cap.read()
        if ret:
            frame_cache[idx] = f.copy()
            return f
        return None

    def reset_zoom():
        global zoom_scale, zoom_cx, zoom_cy
        zoom_scale = 1.0
        zoom_cx    = 0.5
        zoom_cy    = 0.5

    def go_to_frame(idx):
        global frame_orig, boxes
        frame_orig = get_frame(idx)
        if frame_orig is None:
            return False
        h, w = frame_orig.shape[:2]
        lp = str(lbl_dir / f"frame_{idx:06d}.txt")
        # โหลด label เฉพาะเฟรมที่ save ไปแล้วใน session นี้เท่านั้น
        # ป้องกัน label เก่าจากวิดีโออื่นหลุดเข้ามา
        if idx in frames_seen:
            boxes = load_boxes_from_file(lp, w, h)
        else:
            boxes = []
        reset_zoom()
        return True

    # ---- เริ่มต้น ----
    frame_index = 0
    if not go_to_frame(frame_index):
        print("[X] อ่านเฟรมแรกไม่ได้")
        return

    print(f"[OK] วิดีโอ: {total_frames} เฟรม  |  output -> {OUTPUT_DIR}")
    print("     Scroll = Zoom in/out   Middle+drag = Pan   R = Reset zoom")
    print("     S/Space = บันทึก   D/-> = ข้าม   A/<- = ย้อนกลับ   Z = ลบกรอบล่าสุด   Q/ESC = ออก")

    while True:
        if frame_orig is None:
            print("[OK] วิดีโอหมดแล้ว")
            break

        # อัพเดต display size จริง (user resize window)
        try:
            rect = cv2.getWindowImageRect(WINDOW)
            disp_w = max(rect[2], 320)
            disp_h = max(rect[3], 240)
        except Exception:
            disp_w, disp_h = 1280, 720
        mouse_param[0] = disp_w
        mouse_param[1] = disp_h

        vis = draw_overlay(frame_orig, frame_index, total_frames, saved_count, disp_w, disp_h)
        cv2.imshow(WINDOW, vis)
        key = cv2.waitKey(20) & 0xFF

        # ---- Quit ----
        if key in (27, ord('q')):
            print("ออกจากโปรแกรม")
            break

        # ---- Undo box ----
        elif key == ord('z'):
            if boxes:
                boxes.pop()

        # ---- Reset zoom ----
        elif key == ord('r'):
            reset_zoom()

        # ---- Save + Next ----
        elif key in (ord('s'), ord(' ')):
            img_path, lbl_path = save_frame(frame_orig, frame_index, boxes)
            frames_seen[frame_index] = (img_path, lbl_path, list(boxes))
            saved_count += 1
            print(f"  [saved] frame {frame_index:06d}  boxes={len(boxes)}  -> {img_path}")

            next_idx = frame_index + EXTRACT_EVERY_N
            if next_idx >= total_frames:
                print("[OK] ถึงเฟรมสุดท้ายแล้ว")
                break
            frame_index = next_idx
            go_to_frame(frame_index)

        # ---- Skip (ไม่ save) ----
        elif key in (ord('d'), 83):   # 83 = arrow right
            next_idx = frame_index + EXTRACT_EVERY_N
            if next_idx >= total_frames:
                print("[OK] ถึงเฟรมสุดท้ายแล้ว")
                break
            frame_index = next_idx
            go_to_frame(frame_index)

        # ---- Back ----
        elif key in (ord('a'), 81):   # 81 = arrow left
            prev_idx = max(frame_index - EXTRACT_EVERY_N, 0)
            frame_index = prev_idx
            go_to_frame(frame_index)
            print(f"  [back] frame {frame_index:06d}  boxes={len(boxes)}")

    cap.release()
    cv2.destroyAllWindows()

    write_yaml()
    print(f"\n[OK] เสร็จ! บันทึกทั้งหมด {saved_count} เฟรม")
    print(f"     images -> {img_dir}")
    print(f"     labels -> {lbl_dir}")
    print(f"\n--- Fine-tune command ---")
    print(f"yolo detect train data={OUTPUT_DIR}/data.yaml model=best.pt epochs=50 imgsz=640 batch=16")


if __name__ == "__main__":
    main()

[OK] วิดีโอ: 4024 เฟรม  |  output -> real_data
     Scroll = Zoom in/out   Middle+drag = Pan   R = Reset zoom
     S/Space = บันทึก   D/-> = ข้าม   A/<- = ย้อนกลับ   Z = ลบกรอบล่าสุด   Q/ESC = ออก
  [saved] frame 000000  boxes=18  -> real_data\images\train\frame_000000.jpg
  [back] frame 000060  boxes=0
  [back] frame 000000  boxes=18
  [back] frame 000120  boxes=0
  [back] frame 000060  boxes=0
  [back] frame 000000  boxes=18
  [saved] frame 000180  boxes=17  -> real_data\images\train\frame_000180.jpg
  [back] frame 000240  boxes=0
  [back] frame 000180  boxes=17
  [back] frame 000300  boxes=0
  [back] frame 000240  boxes=0
  [back] frame 000180  boxes=17
  [back] frame 000300  boxes=0
  [back] frame 000240  boxes=0
  [back] frame 000180  boxes=17
  [back] frame 000480  boxes=0
  [back] frame 000420  boxes=0
  [back] frame 000360  boxes=0
  [back] frame 000300  boxes=0
  [back] frame 000240  boxes=0
  [back] frame 000180  boxes=17
  [back] frame 000420  boxes=0
  [back] frame 000360  

In [2]:
import cv2
import os
from pathlib import Path

# ===================== CONFIG =====================
IMG_DIR = r"real_data/train/images"
LBL_DIR = r"real_data/train/labels"
# ==================================================

img_files = sorted(Path(IMG_DIR).glob("*.jpg"))

# ============================================================
# Step 1: ลบภาพที่ไม่มี label โดยอัตโนมัติก่อนเลย
# ============================================================
no_label = [p for p in img_files if not (Path(LBL_DIR) / (p.stem + ".txt")).exists()]

if no_label:
    print(f"[!] พบภาพที่ไม่มี label {len(no_label)} ไฟล์ → ลบออกอัตโนมัติ")
    for p in no_label:
        os.remove(p)
        print(f"    [deleted] {p.name}")
else:
    print("[OK] ไม่มีภาพที่ไม่มี label")

# โหลดลิสต์ใหม่หลังลบ
img_files = sorted(Path(IMG_DIR).glob("*.jpg"))
print(f"[OK] เหลือภาพทั้งหมด {len(img_files)} ไฟล์\n")

# ============================================================
# Step 2: Preview ตรวจสอบ bounding box ทีละภาพ
# ============================================================
print("Preview: Space/อื่นๆ = ถัดไป  |  D = ลบภาพนี้  |  ESC = ออก\n")

for i, img_path in enumerate(img_files):
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    h, w = img.shape[:2]

    lbl_path = Path(LBL_DIR) / (img_path.stem + ".txt")
    box_count = 0

    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, cx, cy, bw, bh = map(float, parts)
            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            x2 = int((cx + bw / 2) * w)
            y2 = int((cy + bh / 2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 80), 2)
            cv2.putText(img, "fish", (x1, max(y1 - 6, 12)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 80), 1)
            box_count += 1

    # Header
    cv2.rectangle(img, (0, 0), (w, 38), (20, 20, 20), -1)
    info = f"{i+1}/{len(img_files)}  |  {img_path.name}  |  Boxes: {box_count}  |  [Space]=Next  [D]=Delete  [ESC]=Quit"
    cv2.putText(img, info, (8, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1)

    cv2.imshow("Label Preview", img)
    key = cv2.waitKey(0) & 0xFF

    if key == 27:   # ESC
        print("ออกจาก preview")
        break
    elif key == ord('d'):
        os.remove(img_path)
        os.remove(lbl_path)
        print(f"  [deleted] {img_path.name}")

cv2.destroyAllWindows()
remaining = len(list(Path(IMG_DIR).glob("*.jpg")))
print(f"\n[OK] เสร็จ! เหลือภาพทั้งหมด {remaining} ไฟล์")

[OK] ไม่มีภาพที่ไม่มี label
[OK] เหลือภาพทั้งหมด 180 ไฟล์

Preview: Space/อื่นๆ = ถัดไป  |  D = ลบภาพนี้  |  ESC = ออก

ออกจาก preview

[OK] เสร็จ! เหลือภาพทั้งหมด 180 ไฟล์


In [10]:
import shutil
from pathlib import Path

# ===================== CONFIG =====================
DATASET_DIR = r"real_data"
# ==================================================

base = Path(DATASET_DIR)

# โครงสร้างเดิม
old_img = base / "images" / "train"
old_lbl = base / "labels" / "train"

# โครงสร้างใหม่
new_img = base / "train" / "images"
new_lbl = base / "train" / "labels"

# ตรวจก่อนว่ามีโครงสร้างเดิมอยู่
if not old_img.exists():
    print(f"[X] ไม่พบ {old_img}")
    print("    โครงสร้างอาจถูกย้ายไปแล้ว หรือ path ผิด")
    exit()

# สร้างโฟลเดอร์ใหม่
new_img.mkdir(parents=True, exist_ok=True)
new_lbl.mkdir(parents=True, exist_ok=True)

# ย้ายไฟล์
imgs = list(old_img.glob("*"))
lbls = list(old_lbl.glob("*"))

for f in imgs:
    shutil.move(str(f), new_img / f.name)
for f in lbls:
    shutil.move(str(f), new_lbl / f.name)

# ลบโฟลเดอร์เก่า
shutil.rmtree(base / "images")
shutil.rmtree(base / "labels")

print(f"[OK] ย้ายภาพ  {len(imgs)} ไฟล์  ->  {new_img}")
print(f"[OK] ย้าย label {len(lbls)} ไฟล์  ->  {new_lbl}")

# อัพเดต data.yaml
yaml_path = base / "data.yaml"
content = (
    f"path: {DATASET_DIR}\n"
    f"train: train/images\n"
    f"val:   valid/images\n"
    f"test:  test/images\n\n"
    f"nc: 1\n"
    f"names: ['fish']\n"
)
with open(yaml_path, "w") as f:
    f.write(content)

print(f"[OK] data.yaml อัพเดตแล้ว")
print(f"\nโครงสร้างใหม่:")
for p in sorted(base.rglob("*")):
    if p.is_dir():
        indent = "  " * (len(p.relative_to(base).parts) - 1)
        print(f"  {indent}{p.name}/")

[OK] ย้ายภาพ  41 ไฟล์  ->  real_data\train\images
[OK] ย้าย label 41 ไฟล์  ->  real_data\train\labels
[OK] data.yaml อัพเดตแล้ว

โครงสร้างใหม่:
  train/
    images/
    labels/


# Augmentation

In [1]:
import cv2
import numpy as np
import shutil
from pathlib import Path

# ===================== CONFIG =====================
DATASET_DIR  = r"real_data"
AUG_PER_IMAGE = 5   # สร้างภาพใหม่กี่ภาพต่อ 1 ภาพต้นฉบับ
# ==================================================

src_img = Path(DATASET_DIR) / "train" / "images"
src_lbl = Path(DATASET_DIR) / "train" / "labels"

img_files = sorted(src_img.glob("*.jpg"))
print(f"[OK] พบภาพต้นฉบับ {len(img_files)} ภาพ")
print(f"     จะสร้างเพิ่ม {len(img_files) * AUG_PER_IMAGE} ภาพ")
print(f"     รวมทั้งหมด {len(img_files) * (AUG_PER_IMAGE + 1)} ภาพ\n")


# ============================================================
# Augmentation functions
# ============================================================

def aug_brightness(img):
    """ปรับความสว่าง/contrast สุ่ม"""
    alpha = np.random.uniform(0.5, 1.5)   # contrast
    beta  = np.random.randint(-40, 40)     # brightness
    return np.clip(alpha * img.astype(np.float32) + beta, 0, 255).astype(np.uint8)


def aug_flip_h(img, boxes):
    """Flip แนวนอน + ปรับ boxes"""
    flipped = cv2.flip(img, 1)
    new_boxes = []
    for b in boxes:
        cls, cx, cy, bw, bh = b
        new_boxes.append((cls, 1.0 - cx, cy, bw, bh))
    return flipped, new_boxes


def aug_blur(img):
    """Gaussian blur จำลองภาพใต้น้ำขุ่น"""
    k = np.random.choice([3, 5, 7])
    return cv2.GaussianBlur(img, (k, k), 0)


def aug_hsv(img):
    """เปลี่ยน hue/saturation จำลองสีน้ำต่างกัน"""
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 0] = (hsv[:, :, 0] + np.random.randint(-15, 15)) % 180
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * np.random.uniform(0.7, 1.3), 0, 255)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * np.random.uniform(0.7, 1.3), 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)


def aug_noise(img):
    """เพิ่ม Gaussian noise"""
    noise = np.random.normal(0, np.random.uniform(5, 20), img.shape).astype(np.float32)
    return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)


def aug_crop(img, boxes, crop_ratio=0.85):
    """Random crop แล้ว resize กลับ กรอบที่หลุดออกไปจะถูกตัดทิ้ง"""
    h, w = img.shape[:2]
    crop_w = int(w * crop_ratio)
    crop_h = int(h * crop_ratio)
    x_off  = np.random.randint(0, w - crop_w)
    y_off  = np.random.randint(0, h - crop_h)

    cropped = img[y_off:y_off+crop_h, x_off:x_off+crop_w]
    resized = cv2.resize(cropped, (w, h))

    new_boxes = []
    for b in boxes:
        cls, cx, cy, bw, bh = b
        # แปลงกลับเป็น pixel
        px1 = (cx - bw/2) * w
        py1 = (cy - bh/2) * h
        px2 = (cx + bw/2) * w
        py2 = (cy + bh/2) * h
        # ปรับตาม crop offset
        px1 = (px1 - x_off) / crop_w
        py1 = (py1 - y_off) / crop_h
        px2 = (px2 - x_off) / crop_w
        py2 = (py2 - y_off) / crop_h
        # ตัดกรอบที่หลุดออกไป
        px1, px2 = np.clip([px1, px2], 0, 1)
        py1, py2 = np.clip([py1, py2], 0, 1)
        if px2 - px1 > 0.05 and py2 - py1 > 0.05:
            ncx = (px1 + px2) / 2
            ncy = (py1 + py2) / 2
            nbw = px2 - px1
            nbh = py2 - py1
            new_boxes.append((cls, ncx, ncy, nbw, nbh))
    return resized, new_boxes


# รายการ augmentation ทั้งหมด (ไม่รวม flip เพราะ boxes ต้องปรับแยก)
AUG_FUNCS = [aug_brightness, aug_blur, aug_hsv, aug_noise]


# ============================================================
# อ่าน label
# ============================================================
def read_label(lbl_path):
    boxes = []
    if not lbl_path.exists():
        return boxes
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                boxes.append(tuple(map(float, parts)))
    return boxes


def write_label(lbl_path, boxes):
    with open(lbl_path, "w") as f:
        for b in boxes:
            f.write(f"{int(b[0])} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\n")


# ============================================================
# Main loop
# ============================================================
created = 0

for img_path in img_files:
    lbl_path = src_lbl / (img_path.stem + ".txt")
    img      = cv2.imread(str(img_path))
    boxes    = read_label(lbl_path)

    if img is None:
        print(f"  [skip] อ่านภาพไม่ได้: {img_path.name}")
        continue

    aug_list = []

    # 1. Flip แนวนอน (boxes ต้องแปลงพิเศษ)
    aug_list.append(aug_flip_h(img, boxes))

    # 2-5. สุ่ม augmentation จาก AUG_FUNCS ที่เหลือ
    remaining = AUG_PER_IMAGE - 1
    chosen    = np.random.choice(len(AUG_FUNCS), size=remaining, replace=True)
    for idx in chosen:
        fn     = AUG_FUNCS[idx]
        result = fn(img)
        aug_list.append((result, boxes))

    # 6. crop (โอกาส 50%)
    if np.random.rand() < 0.5:
        aug_list.append(aug_crop(img, boxes))

    # บันทึกภาพ augmented
    for i, (aug_img, aug_boxes) in enumerate(aug_list[:AUG_PER_IMAGE]):
        new_stem     = f"{img_path.stem}_aug{i+1}"
        new_img_path = src_img / f"{new_stem}.jpg"
        new_lbl_path = src_lbl / f"{new_stem}.txt"
        cv2.imwrite(str(new_img_path), aug_img)
        write_label(new_lbl_path, aug_boxes)
        created += 1

    print(f"  [aug] {img_path.name}  +{min(len(aug_list), AUG_PER_IMAGE)} ภาพ")

total_now = len(list(src_img.glob("*.jpg")))
print(f"\n[OK] สร้างภาพใหม่ {created} ภาพ")
print(f"     train ตอนนี้มีทั้งหมด {total_now} ภาพ")

[OK] พบภาพต้นฉบับ 30 ภาพ
     จะสร้างเพิ่ม 150 ภาพ
     รวมทั้งหมด 180 ภาพ

  [aug] frame_000030.jpg  +5 ภาพ
  [aug] frame_000180.jpg  +5 ภาพ
  [aug] frame_000240.jpg  +5 ภาพ
  [aug] frame_000300.jpg  +5 ภาพ
  [aug] frame_000480.jpg  +5 ภาพ
  [aug] frame_001140.jpg  +5 ภาพ
  [aug] frame_001620.jpg  +5 ภาพ
  [aug] frame_001710.jpg  +5 ภาพ
  [aug] frame_001800.jpg  +5 ภาพ
  [aug] frame_002010.jpg  +5 ภาพ
  [aug] frame_002160.jpg  +5 ภาพ
  [aug] frame_002220.jpg  +5 ภาพ
  [aug] frame_002280.jpg  +5 ภาพ
  [aug] frame_002310.jpg  +5 ภาพ
  [aug] frame_002340.jpg  +5 ภาพ
  [aug] frame_002400.jpg  +5 ภาพ
  [aug] frame_002490.jpg  +5 ภาพ
  [aug] frame_002520.jpg  +5 ภาพ
  [aug] frame_002580.jpg  +5 ภาพ
  [aug] frame_002700.jpg  +5 ภาพ
  [aug] frame_002790.jpg  +5 ภาพ
  [aug] frame_002820.jpg  +5 ภาพ
  [aug] frame_002880.jpg  +5 ภาพ
  [aug] frame_002940.jpg  +5 ภาพ
  [aug] frame_002970.jpg  +5 ภาพ
  [aug] frame_003060.jpg  +5 ภาพ
  [aug] frame_003180.jpg  +5 ภาพ
  [aug] frame_003300.jpg  +5 ภาพ
